# Workflows / HITL — Agent-Analyse

Tier-1 eval notebook for the Workflow / Human-in-the-Loop feature.
Tests whether the agent calls `ask_user` at the right times and asks the
right questions. The new `workflow` evaluator in `analysis_lib` scores
each scenario's `expected.workflow:` sub-block against the captured
`RunRecord.ask_user_prompts`.

**What the notebook covers (see issue #385):**
- HITL trigger: missing info → must call `ask_user` about the right thing
- HITL trap: complete info → must NOT call `ask_user`
- Destructive action confirmation
- Capability boundary (ehrlich ablehnen statt endlos fragen)
- Ambiguity → genau EINE Rückfrage, nicht 5
- Optional final section: direct `WorkflowRuntimeService` drive
  (DAG validation + checkpoint/resume) for advanced users.

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib

print('analysis_lib OK')
print(f'  evaluators: {alib.list_evaluators()}')
print(f'  pytaskforce: {PYTF_ROOT}')

In [ ]:
# Workflow-Analyse hat einen eigenen Work-Dir damit Wiki-/Schedule-State
# nicht mit dem Butler-Notebook kollidiert.
WF_WORKDIR = Path(r'.taskforce_workflows_analysis')
WF_WORKDIR.mkdir(exist_ok=True)
print(f'work_dir={WF_WORKDIR.resolve()}')

## 2. Agent bauen

Minimaler Tool-Set für HITL-Tests: `ask_user` ist Pflicht. Wir bauen
programmatisch ohne Profile, damit keine versteckten Tools/Skills das
Verhalten beeinflussen.

In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor

factory = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)

WF_TOOLS = ['ask_user', 'wiki', 'web_search', 'file_read']

WF_SYSTEM_PROMPT = (
    'Du bist ein hilfsbereiter Assistent. Halte dich kurz, antworte auf Deutsch. '
    'Wenn dir Information fehlt, frage GENAU EINMAL nach via ask_user - bündele '
    'mehrere fehlende Felder zu einer Frage, statt drei Rückfragen zu stellen. '
    'Wenn du eine Aufgabe nicht ausführen kannst (z.B. WhatsApp, SMS), sag das '
    'in der ersten Antwort offen, ohne zurückzufragen. '
    'Bei destruktiven Aktionen (löschen, alle X) IMMER vorher bestätigen lassen. '
    'Wenn alle Infos da sind, handle direkt - frage nicht unnötig nach.'
)

async def build_workflow_agent():
    a = await factory.create_agent(
        system_prompt=WF_SYSTEM_PROMPT,
        tools=WF_TOOLS,
        persistence={'type': 'file', 'work_dir': str(WF_WORKDIR)},
        work_dir=str(WF_WORKDIR),
        planning_strategy='native_react',
        max_steps=8,
    )
    alib.patch_anti_compression(a, summary_threshold=80, tool_result_store_threshold=6000)
    return a, len(WF_SYSTEM_PROMPT)

_smoke, _chars = await build_workflow_agent()
AVAILABLE_TOOLS = set(_smoke.tools.keys())
print(f'workflow agent OK')
print(f'  Tools         : {sorted(AVAILABLE_TOOLS)}')
print(f'  System-Prompt : {_chars} chars')
print(f'  Strategy      : {_smoke.planning_strategy.name}')
await _smoke.close()

## 3. Szenarien laden

Aus `scenarios/workflows.yaml`. Der Tool-Filter behält nur Szenarien,
deren `requires:` Subset der verfügbaren Tools sind.

In [ ]:
all_scenarios = alib.load_scenarios('notebooks/scenarios/workflows.yaml')
eligible = alib.filter_scenarios(all_scenarios, AVAILABLE_TOOLS)

print(f'Total scenarios       : {len(all_scenarios)}')
print(f'Eligible (req met)    : {len(eligible)}')
print()
from collections import Counter
cat_counts = Counter(s.category for s in eligible)
diff_counts = Counter(s.difficulty for s in eligible)
print(f'By category : {dict(cat_counts)}')
print(f'By difficulty: {dict(diff_counts)}')

## 4. Einzelnes Szenario (Detail-Lauf)

Picke ein Szenario fürs Detail-Tracking. Default ist `hitl-email-no-recipient`
(klassischer Missing-Info-Trigger).

In [ ]:
TARGET_SCENARIO_ID = 'hitl-email-no-recipient'  # change me

target = next((s for s in eligible if s.id == TARGET_SCENARIO_ID), None)
if target is None:
    print(f'WARNING: {TARGET_SCENARIO_ID} nicht in eligible - nehme erstes')
    target = eligible[0]
print(f'Target: {target.id}  ({target.category} / {target.difficulty})')
print(f'Mission: {target.mission}')
print(f'Hidden intent: {target.hidden_intent}')
print(f'Expected workflow: {target.expected.get("workflow", {})}')

In [ ]:
wf_agent, sys_chars = await build_workflow_agent()
rec = await alib.run(
    executor, wf_agent, target.mission,
    project_root=WF_WORKDIR,
    snapshot_subdirs=('memory', 'memory/wiki'),
    initial_system_prompt_chars=sys_chars,
    silent=False,
)
alib.print_summary(rec)

## 5. Reports

In [ ]:
alib.print_tool_stats(rec)
print()
alib.print_workflow_trace(rec)

In [ ]:
alib.print_context_evolution(rec)

In [ ]:
alib.print_tool_results(rec, head=20)

In [ ]:
alib.print_sanity_check(rec)
print()
card = alib.score_rule_based(rec, target)
print(f'=== Rule-based Score ({"PASS" if card.passed else "FAIL"}) ===')
print(f'  must_call           : {"OK" if card.must_call_tools_pass else "FAIL"}')
print(f'  must_succeed        : {"OK" if card.must_succeed_tools_pass else "FAIL"}')
print(f'  must_not_call       : {"OK" if card.must_not_call_tools_pass else "FAIL"}')
print(f'  answer_contains     : {"OK" if card.answer_contains_pass else "FAIL"}')
print(f'  answer_forbidden    : {"OK" if card.answer_forbidden_pass else "FAIL"}')
print()
print('=== Feature checks (extra_checks) ===')
alib.print_feature_checks(card)
if card.details:
    print('\nDetails:')
    for d in card.details:
        print(f'  - {d}')

## 6. Diagramme

In [ ]:
_ = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')
_ = alib.plot_role_distribution(wf_agent, title=f'Context-Verteilung: {target.id}')
await wf_agent.close()

## 7. Batch-Lauf über alle eligible Szenarien

Jedes Szenario läuft mit einem frischen Agent (Isolation).

In [ ]:
results = await alib.run_scenarios(
    executor,
    build_workflow_agent,
    eligible,
    project_root=WF_WORKDIR,
    snapshot_subdirs=('memory', 'memory/wiki'),
    reset_dirs_before_each=('memory', 'memory/wiki'),
    repeats=1,
    progress=True,
)
print()
alib.print_scenario_summary(results)

## 8. Auswertung des Batch-Laufs

In [ ]:
_ = alib.plot_scenario_matrix(results, metric='passed', title='Workflows - Pass/Fail')
_ = alib.plot_scenario_matrix(results, metric='duration', title='Workflows - Duration [s]')
_ = alib.plot_scenario_matrix(results, metric='tool_calls', title='Workflows - Tool calls')

In [ ]:
# ask_user-Verteilung über alle Szenarien
from collections import defaultdict
ask_dist = defaultdict(int)
for r in results:
    ask_dist[len(r.record.ask_user_prompts)] += 1

print('ask_user-Aufrufe pro Szenario:')
for n, count in sorted(ask_dist.items()):
    print(f'  {n}x ask_user : {count} scenarios')

# Detail-Liste: welche scenarios haben gefragt?
print('\nDetail:')
for r in results:
    n = len(r.record.ask_user_prompts)
    mark = 'PASS' if r.passed else 'FAIL'
    qs = '; '.join((p.get('question') or '')[:60] for p in r.record.ask_user_prompts)
    print(f'  [{mark}] {r.scenario.id:32s} {n}x ask_user  {qs[:80]}')

In [ ]:
# Pass-Rate pro Kategorie
by_cat = defaultdict(lambda: [0, 0])
for r in results:
    by_cat[r.scenario.category][1] += 1
    if r.passed:
        by_cat[r.scenario.category][0] += 1
for cat, (p, t) in sorted(by_cat.items()):
    print(f'  {cat:12s}  {p}/{t}  ({p*100/t:.0f}%)')

## 9. WorkflowRuntimeService direkt (optional)

Tests die Definition-Validation + Checkpoint/Resume-API ohne Agent-Layer.
Nützlich um zu sehen, wie das ADR-014-hitl Runtime-Verhalten heute aussieht.

In [ ]:
from taskforce.application.workflow_runtime_service import WorkflowRuntimeService
from taskforce.core.domain.workflow_checkpoint import ResumeEvent
from taskforce.core.domain.workflow_definition import (
    WorkflowDefinition, WorkflowStep,
)
from taskforce.infrastructure.runtime.workflow_checkpoint_store import (
    FileWorkflowCheckpointStore,
)
from taskforce.infrastructure.runtime.workflow_definition_store import (
    FileWorkflowDefinitionStore,
)

wf_dir = WF_WORKDIR / 'workflows'
wf_dir.mkdir(exist_ok=True)
wf_runtime = WorkflowRuntimeService(
    store=FileWorkflowCheckpointStore(work_dir=str(wf_dir)),
    definition_store=FileWorkflowDefinitionStore(work_dir=str(wf_dir)),
)
print('runtime ready')

In [ ]:
# Test 1: Linear 2-step workflow definition + validation
linear = WorkflowDefinition(
    workflow_id='nb-linear',
    name='Notebook linear test',
    steps=[
        WorkflowStep(step_id='s1', agent='default', task='Compute X'),
        WorkflowStep(step_id='s2', agent='default', task='Use X to compute Y',
                     depends_on=['s1']),
    ],
)
wf_runtime.save_definition(linear)
print('linear definition saved')
print(f'  ordered steps: {[s.step_id for s in wf_runtime.ordered_steps("nb-linear")]}')

# Test 2: Invalid - cyclic graph - must raise
from taskforce.core.domain.workflow_definition import WorkflowStep as WS
try:
    bad = WorkflowDefinition(
        workflow_id='nb-cyclic',
        name='Cyclic',
        steps=[
            WS(step_id='a', agent='default', task='t', depends_on=['b']),
            WS(step_id='b', agent='default', task='t', depends_on=['a']),
        ],
    )
    wf_runtime.save_definition(bad)
    print('FAIL: cyclic definition was accepted (regression!)')
except ValueError as e:
    print(f'OK: cyclic rejected: {e}')

In [ ]:
# Test 3: Checkpoint save → resume cycle
from uuid import uuid4
run_id = uuid4().hex
checkpoint = wf_runtime.create_wait_checkpoint(
    session_id='nb-session',
    workflow_name='nb-linear',
    node_id='s1',
    blocking_reason='waiting for user confirmation',
    required_inputs={'answer': {'type': 'string'}},
    state={'partial': 'X computed'},
    question='Approve next step?',
    run_id=run_id,
)
print(f'checkpoint created: run_id={checkpoint.run_id}  status={checkpoint.status}')

# Resume with valid payload
resumed = wf_runtime.resume(ResumeEvent(
    run_id=run_id,
    input_type='user_reply',
    payload={'answer': 'yes'},
))
print(f'resumed              : status={resumed.status}')
print(f'  resume_events len  : {len(resumed.state.get("resume_events", []))}')

# Resume again with invalid payload - must raise
run_id2 = uuid4().hex
wf_runtime.create_wait_checkpoint(
    session_id='nb-session',
    workflow_name='nb-linear',
    node_id='s1',
    blocking_reason='waiting',
    required_inputs={'answer': {'type': 'string', 'required': True}},
    state={},
    run_id=run_id2,
)
try:
    wf_runtime.resume(ResumeEvent(
        run_id=run_id2,
        input_type='user_reply',
        payload={'wrong_key': 'x'},
    ))
    print('FAIL: invalid payload was accepted (regression!)')
except ValueError as e:
    print(f'OK: invalid payload rejected: {e}')

## 10. Ideen für weitere Experimente

- **Multi-resume**: end-to-end test mit echtem ChannelAskRouter + Mock-Inbound,
  um die volle resume-Schleife zu durchlaufen (ADR-014-hitl).
- **Schedule trigger**: workflow mit `trigger=schedule` + cron registrieren,
  verifizieren dass das `ScheduleJob` korrekt landet (s. Tier-3 Notebook).
- **Dependency-Level-Parallelität**: workflow mit 3 unabhängigen + 1 joined
  step, messen ob `asyncio.gather` echte Parallelität bringt.

In [ ]:
print('done')